# RNA-seq Preprocessing Pipeline: SARS-CoV-2 Age Comparison in Mice
**Study:** Transcriptomic response to SARS-CoV-2 infection in young vs. mid-age mice  
**Pipeline:** SRA download → FastQC → Cutadapt trimming → HISAT2 alignment → featureCounts quantification  
**Reference genome:** *Mus musculus* mm10 (GRCm38)

---

## Overview

This notebook implements a complete RNA-seq preprocessing pipeline, taking raw sequencing reads from the NCBI Sequence Read Archive (SRA) all the way through to a gene-level read count matrix ready for differential expression analysis.

**Biological context:** SARS-CoV-2 infection causes markedly different outcomes depending on age — older individuals experience more severe disease. This pipeline processes RNA-seq data from a mouse model comparing the transcriptomic response to SARS-CoV-2 infection between young and mid-age mice, alongside PBS (vehicle control) groups for each age group. The resulting count matrix captures how gene expression differs between:
- Young mice + SARS-CoV-2 vs. young mice + PBS (infection effect in young animals)
- Mid-age mice + SARS-CoV-2 vs. mid-age mice + PBS (infection effect in older animals)
- Across age groups (age effect on the transcriptomic response to infection)

**SRA accessions:**

| Sample | SRA Accession |
|---|---|
| Young + SARS-CoV-2 | SRR24206824 |
| Mid-age + SARS-CoV-2 | SRR24206825 |
| Young + PBS (control) | SRR24206826 |
| Mid-age + PBS (control) | SRR24206827 |

**Note:** each sample is subsampled to 5 million reads to make this pipeline feasible within Colab's compute limits. A full analysis would use all available reads.

**Pipeline steps:**
1. Download raw FASTQ files from NCBI SRA
2. Assess read quality with FastQC
3. Trim adapters and low-quality bases with Cutadapt
4. Re-run FastQC to confirm trimming improved quality
5. Align trimmed reads to mm10 with HISAT2 (splice-aware aligner)
6. Quantify gene-level read counts with featureCounts
7. Load and inspect the resulting count matrix

## 1. Setup — Install Tools

We use four standard bioinformatics tools, each handling a distinct stage of the pipeline:

| Tool | Stage | Why this tool |
|---|---|---|
| **SRA Toolkit** | Data download | NCBI's official tool for downloading sequencing data from SRA |
| **FastQC** | Quality control | Industry-standard QC report for Illumina sequencing data |
| **Cutadapt** | Read trimming | Removes adapter sequences and low-quality bases from reads |
| **HISAT2** | Alignment | Splice-aware aligner for RNA-seq reads; handles reads spanning exon-exon junctions |
| **featureCounts** | Quantification | Assigns aligned reads to genomic features (genes/exons) using a GTF annotation |

In [ ]:
# Install SRA Toolkit (NCBI's tool for downloading from Sequence Read Archive)
!wget -q https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/3.1.0/sratoolkit.3.1.0-ubuntu64.tar.gz
!tar -xzf sratoolkit.3.1.0-ubuntu64.tar.gz
!chmod +x sratoolkit.3.1.0-ubuntu64/bin/fastq-dump
print("SRA Toolkit installed.")

In [ ]:
# Install Java (required by FastQC), then download and unzip FastQC
!sudo apt-get install -y default-jre -q
!wget -q https://www.bioinformatics.babraham.ac.uk/projects/fastqc/fastqc_v0.11.9.zip
!unzip -q fastqc_v0.11.9.zip
!chmod +x FastQC/fastqc

# Install Cutadapt (Python-based trimmer)
!pip install cutadapt -q

# Install HISAT2 and featureCounts (part of the Subread package)
!apt-get install -y hisat2 subread -q

print("FastQC, Cutadapt, HISAT2, and featureCounts installed.")
!./FastQC/fastqc --version

## 2. Download Raw Sequencing Reads from NCBI SRA

We use `fastq-dump` with `-X 5000000` to download the first 5 million reads from each SRA accession. For a full analysis, omit `-X` to download all reads (this would take significantly longer and require more disk space).

After downloading, we rename files to descriptive sample names so they're easy to identify throughout the pipeline.

In [ ]:
# Download 5 million reads per sample (subsampled for computational feasibility)
print("Downloading SRR24206824 (Young + SARS-CoV-2)...")
!sratoolkit.3.1.0-ubuntu64/bin/fastq-dump -X 5000000 SRR24206824

print("Downloading SRR24206825 (Mid-age + SARS-CoV-2)...")
!sratoolkit.3.1.0-ubuntu64/bin/fastq-dump -X 5000000 SRR24206825

print("Downloading SRR24206826 (Young + PBS control)...")
!sratoolkit.3.1.0-ubuntu64/bin/fastq-dump -X 5000000 SRR24206826

print("Downloading SRR24206827 (Mid-age + PBS control)...")
!sratoolkit.3.1.0-ubuntu64/bin/fastq-dump -X 5000000 SRR24206827

print("All downloads complete.")

In [ ]:
# Rename files to descriptive sample names for clarity downstream
!mv SRR24206824.fastq Young_SARS-CoV-2.fastq
!mv SRR24206825.fastq MidAge_SARS-CoV-2.fastq
!mv SRR24206826.fastq Young_PBS.fastq
!mv SRR24206827.fastq MidAge_PBS.fastq

# Confirm files are present and check sizes
!ls -lh *.fastq

## 3. Quality Control — FastQC (Pre-Trimming)

FastQC generates a comprehensive HTML quality report for each FASTQ file, summarizing:
- Per-base sequence quality (Phred scores across read position)
- Per-sequence quality score distribution
- Sequence length distribution
- GC content distribution
- Adapter content and overrepresented sequences

Running FastQC **before trimming** establishes a baseline — we can then compare against the post-trimming report to confirm that quality issues were addressed.

In [ ]:
# Run FastQC on all four raw FASTQ files
!./FastQC/fastqc Young_SARS-CoV-2.fastq MidAge_SARS-CoV-2.fastq \
                 Young_PBS.fastq MidAge_PBS.fastq

!ls -lh *_fastqc.html

In [ ]:
# Download FastQC HTML reports to inspect locally
# (In Colab, HTML files render in a browser after downloading)
from google.colab import files

for sample in ['Young_SARS-CoV-2', 'MidAge_SARS-CoV-2', 'Young_PBS', 'MidAge_PBS']:
    try:
        files.download(f'{sample}_fastqc.html')
    except Exception as e:
        print(f"Could not download {sample}_fastqc.html: {e}")

## 4. Read Trimming — Cutadapt

Raw reads often contain adapter sequences (from the sequencing library preparation) and low-quality bases (particularly at the 3' end of reads, where error rates tend to increase). We use Cutadapt to:

- **`--cut 200`** — hard-trim the first 200 bases from each read. This removes a low-complexity region at the 5' end characteristic of this particular library preparation protocol
- **`-q 28`** — trim low-quality bases from the 3' end where the Phred quality score drops below 28 (approximately 0.16% error rate)

**Note:** trimming parameters are dataset-specific and should be informed by the FastQC reports. The values used here reflect the characteristics of this particular dataset.

In [ ]:
# Trim all four samples
print("Trimming Young_SARS-CoV-2...")
!cutadapt --cut 200 -q 28 -o Trimmed_Young_SARS-CoV-2.fastq Young_SARS-CoV-2.fastq

print("\nTrimming MidAge_SARS-CoV-2...")
!cutadapt --cut 200 -q 28 -o Trimmed_MidAge_SARS-CoV-2.fastq MidAge_SARS-CoV-2.fastq

print("\nTrimming Young_PBS...")
!cutadapt --cut 200 -q 28 -o Trimmed_Young_PBS.fastq Young_PBS.fastq

print("\nTrimming MidAge_PBS...")
!cutadapt --cut 200 -q 28 -o Trimmed_MidAge_PBS.fastq MidAge_PBS.fastq

print("\nAll samples trimmed.")
!ls -lh Trimmed_*.fastq

### 4.1 Post-trimming quality check

Re-running FastQC after trimming lets us confirm that quality issues flagged in the pre-trimming report (adapter content, poor 3' quality) have been resolved.

In [ ]:
# FastQC on trimmed reads
!./FastQC/fastqc Trimmed_Young_SARS-CoV-2.fastq Trimmed_MidAge_SARS-CoV-2.fastq \
                 Trimmed_Young_PBS.fastq Trimmed_MidAge_PBS.fastq

print("Post-trimming FastQC complete.")

## 5. Alignment — HISAT2

**HISAT2** (Hierarchical Indexing for Spliced Alignment of Transcripts) is a splice-aware aligner specifically designed for RNA-seq data. Unlike DNA aligners (like BWA-MEM), HISAT2 understands that RNA-seq reads can span exon-exon junctions — a read that begins in one exon and ends in the next won't align contiguously to the genome unless the aligner accounts for the intervening intron.

We download the pre-built mm10 HISAT2 index from the HISAT2 website rather than building it from scratch, which would require significant compute time and memory.

In [ ]:
# Download the pre-built mm10 HISAT2 genome index
# This avoids building the index from scratch (~1 hour, ~8GB RAM)
print("Downloading mm10 HISAT2 index (this may take several minutes)...")
!wget -q ftp://ftp.ccb.jhu.edu/pub/infphilo/hisat2/data/mm10.tar.gz
!tar -xzf mm10.tar.gz
!ls mm10/
print("mm10 index ready.")

In [ ]:
# Align all four trimmed samples to mm10
# -x: genome index prefix
# -p: number of threads (50 for Colab, adjust if running locally)
# -U: single-end reads (no mate pairing in this dataset)
# -S: output SAM file

print("Aligning Young_SARS-CoV-2...")
!hisat2 -x mm10/genome -p 8 -U Trimmed_Young_SARS-CoV-2.fastq -S Young_SARS-CoV-2.sam

print("\nAligning MidAge_SARS-CoV-2...")
!hisat2 -x mm10/genome -p 8 -U Trimmed_MidAge_SARS-CoV-2.fastq -S MidAge_SARS-CoV-2.sam

print("\nAligning Young_PBS...")
!hisat2 -x mm10/genome -p 8 -U Trimmed_Young_PBS.fastq -S Young_PBS.sam

print("\nAligning MidAge_PBS...")
!hisat2 -x mm10/genome -p 8 -U Trimmed_MidAge_PBS.fastq -S MidAge_PBS.sam

print("\nAll alignments complete.")
!ls -lh *.sam

## 6. Gene Quantification — featureCounts

**featureCounts** (part of the Subread package) counts how many aligned reads overlap each annotated gene in the genome, using a GTF annotation file that describes gene and exon coordinates.

We download the Ensembl GRCm38 release 102 annotation, which matches the mm10 genome assembly used for alignment.

In [ ]:
# Download the mouse gene annotation (GTF format, Ensembl release 102 / GRCm38)
print("Downloading Ensembl mm10 GTF annotation...")
!wget -q https://ftp.ensembl.org/pub/release-102/gtf/mus_musculus/Mus_musculus.GRCm38.102.gtf.gz
!gunzip Mus_musculus.GRCm38.102.gtf.gz
print("Annotation downloaded.")

In [ ]:
# Run featureCounts on all four SAM files simultaneously
# -a: annotation GTF file
# -o: output count matrix
# All SAM files as input — featureCounts produces one column per sample
!featureCounts -a Mus_musculus.GRCm38.102.gtf -o counts.txt *.sam

print("\nfeatureCounts complete. Count matrix saved to counts.txt")
!wc -l counts.txt

## 7. Inspect the Count Matrix

The output `counts.txt` is a tab-delimited file with one row per gene and one column per sample. The first line is a comment with the featureCounts command, the second line is the header. We load it into pandas for inspection.

In [ ]:
import pandas as pd

# Skip the first comment line (featureCounts command echo)
counts_df = pd.read_csv('counts.txt', sep='	', comment='#')
print(f"Count matrix shape: {counts_df.shape}  (genes × samples)")
counts_df.head()

In [ ]:
# Rename sample columns for clarity
# featureCounts uses the full SAM filename as column headers
sample_cols = [c for c in counts_df.columns if c.endswith('.sam')]
rename_map = {
    col: col.replace('.sam', '').replace('Trimmed_', '')
    for col in sample_cols
}
counts_df = counts_df.rename(columns=rename_map)

print("Renamed columns:", [c for c in counts_df.columns if c not in
      ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length']])

In [ ]:
import matplotlib.pyplot as plt

# Total read counts per sample — a quick check for library size differences
sample_names = ['Young_SARS-CoV-2', 'MidAge_SARS-CoV-2', 'Young_PBS', 'MidAge_PBS']
sample_names = [s for s in sample_names if s in counts_df.columns]

if sample_names:
    total_counts = counts_df[sample_names].sum()

    plt.figure(figsize=(8, 4))
    total_counts.plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('Total Mapped Read Counts per Sample')
    plt.ylabel('Total Counts')
    plt.xlabel('Sample')
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

    print("\nTotal counts per sample:")
    print(total_counts.to_string())

In [ ]:
# How many genes have at least 1 count in at least one sample?
if sample_names:
    expressed = (counts_df[sample_names].sum(axis=1) > 0).sum()
    total = len(counts_df)
    print(f"Genes with any counts: {expressed:,} / {total:,} ({expressed/total:.1%})")

    # Save the count matrix for downstream DESeq2/PyDESeq2 analysis
    counts_df.to_csv('gene_counts_SARS_CoV2_age_comparison.csv', index=False)
    print("Count matrix saved to gene_counts_SARS_CoV2_age_comparison.csv")

## 8. Summary

This notebook implemented a complete RNA-seq preprocessing pipeline from raw SRA reads to a gene-level count matrix:

**Pipeline steps completed:**

| Step | Tool | Output |
|---|---|---|
| Data download | SRA Toolkit (`fastq-dump`) | 4 × FASTQ files (5M reads each) |
| Quality control | FastQC | HTML QC reports (pre- and post-trimming) |
| Read trimming | Cutadapt | 4 × trimmed FASTQ files |
| Alignment | HISAT2 | 4 × SAM alignment files |
| Quantification | featureCounts | `counts.txt` — gene × sample count matrix |

**Key decision points documented:**
- `-X 5000000` — subsampled to 5M reads per sample for computational feasibility; full analysis would use all reads
- `--cut 200 -q 28` in Cutadapt — hard 5' trim of 200bp + quality trimming at Phred < 28; parameters selected based on FastQC pre-trim reports
- HISAT2 pre-built mm10 index — avoids ~1 hour of indexing time; ensures alignment to the correct reference version
- Ensembl GRCm38.102 GTF — annotation release matched to the mm10 genome assembly

**Next steps:** the output count matrix (`gene_counts_SARS_CoV2_age_comparison.csv`) is ready for differential expression analysis using PyDESeq2, DESeq2 (R), or edgeR to identify genes significantly up- or down-regulated by SARS-CoV-2 infection, and whether those responses differ between young and mid-age animals. See the companion `differential-gene-expression` repository for an example DESeq2 analysis using a similar RNA-seq count matrix.